In [ ]:
import os
import numpy as np
import nrrd
from skimage import measure, morphology
from scipy import ndimage
import trimesh


# =========================================================
# ⚙️ CONFIGURACIÓN — EDITA AQUÍ
# =========================================================

NRRD_PATH = "/kaggle/working/predicciones_test_nrrd/Paciente32/Paciente32_preds_volume.nrrd"
OUT_DIR   = "/kaggle/working/mallas/Paciente32"

# None = todas las clases; o lista: [10, 11, 5, 6]
LABELS_TO_PROCESS = None


# =========================================================
# 🗂️ NOMBRES Y CONFIGURACIÓN POR CLASE
# =========================================================

LABEL_TO_NAME = {
    1:  "rinones",
    2:  "higado",
    3:  "estomago",
    4:  "pancreas",
    5:  "pulmon_izq",
    6:  "pulmon_dcho",
    7:  "esofago",
    8:  "traquea",
    9:  "tiroides",
    10: "huesos",
    11: "corazon",
    12: "sangre",
    13: "medula_espinal",
    14: "musculo",
    15: "piel",
}

# min_size          → voxels mínimos para considerar componente válida
# keep_largest_cc   → solo componente más grande (False en estructuras múltiples)
# fill_holes        → rellenar huecos internos
# closing_radius    → radio ball binary_closing (0 = desactivado)
# smooth_iterations → iteraciones suavizado Laplaciano (0 = sin suavizado)
# step_size         → paso Marching Cubes (1 = máximo detalle)

LABEL_CONFIG = {
    1:  dict(min_size=200,  keep_largest_cc=False, fill_holes=True,  closing_radius=1, smooth_iterations=8,  step_size=1),  # riñones: 2 componentes
    2:  dict(min_size=1000, keep_largest_cc=True,  fill_holes=True,  closing_radius=1, smooth_iterations=10, step_size=1),  # hígado: compacto
    3:  dict(min_size=500,  keep_largest_cc=True,  fill_holes=True,  closing_radius=2, smooth_iterations=10, step_size=1),  # estómago: closing 2 para eliminar pinchos
    4:  dict(min_size=100,  keep_largest_cc=True,  fill_holes=True,  closing_radius=1, smooth_iterations=8,  step_size=1),  # páncreas: pequeño e irregular
    5:  dict(min_size=1000, keep_largest_cc=True,  fill_holes=True,  closing_radius=1, smooth_iterations=10, step_size=1),  # pulmón izq
    6:  dict(min_size=1000, keep_largest_cc=True,  fill_holes=True,  closing_radius=1, smooth_iterations=10, step_size=1),  # pulmón dcho
    7:  dict(min_size=50,   keep_largest_cc=True,  fill_holes=True,  closing_radius=1, smooth_iterations=5,  step_size=1),  # esófago: tubo fino
    8:  dict(min_size=50,   keep_largest_cc=True,  fill_holes=True,  closing_radius=1, smooth_iterations=5,  step_size=1),  # tráquea: tubo fino
    9:  dict(min_size=50,   keep_largest_cc=False, fill_holes=True,  closing_radius=1, smooth_iterations=5,  step_size=1),  # tiroides: 2 lóbulos
    10: dict(min_size=50,   keep_largest_cc=False, fill_holes=True,  closing_radius=0, smooth_iterations=3,  step_size=1),  # huesos: muchas componentes, sin closing
    11: dict(min_size=1000, keep_largest_cc=True,  fill_holes=True,  closing_radius=2, smooth_iterations=15, step_size=1),  # corazón: closing 2 + más suavizado para eliminar pinchos
    12: dict(min_size=200,  keep_largest_cc=False, fill_holes=True,  closing_radius=1, smooth_iterations=8,  step_size=1),  # sangre: aorta + vasos
    13: dict(min_size=50,   keep_largest_cc=True,  fill_holes=True,  closing_radius=1, smooth_iterations=5,  step_size=1),  # médula: tubo fino
    14: dict(min_size=500,  keep_largest_cc=False, fill_holes=True,  closing_radius=1, smooth_iterations=8,  step_size=1),  # músculo: múltiples grupos
    15: dict(min_size=5000, keep_largest_cc=True,  fill_holes=True,  closing_radius=3, smooth_iterations=10, step_size=2),  # piel: closing agresivo + largest cc para envoltura exterior
}


# =========================================================
# 📂 CARGA VOLUMEN
# =========================================================
def load_nrrd_volume(nrrd_path):
    vol, header = nrrd.read(nrrd_path)
    vol = np.asarray(vol)

    spacing = (2.5,2.5,2.5)
    if "space directions" in header:
        sd = header["space directions"]
        try:
            norms = [float(np.linalg.norm(v)) for v in sd if v is not None and np.linalg.norm(v) > 0]
            if len(norms) == 3:
                spacing = tuple(norms)
        except Exception:
            pass

    return vol, spacing


# =========================================================
# 🧹 POSTPROCESADO
# =========================================================
def postprocess_mask(mask, min_size, keep_largest_cc, fill_holes, closing_radius):
    mask = mask.astype(bool)

    # Closing antes de fill_holes para cerrar conexiones finas
    if closing_radius > 0:
        footprint = morphology.ball(closing_radius)
        mask = morphology.binary_closing(mask, footprint=footprint)

    if fill_holes:
        mask = ndimage.binary_fill_holes(mask)

    mask = morphology.remove_small_objects(mask, min_size=min_size)

    if keep_largest_cc:
        labeled, num = ndimage.label(mask)
        if num > 0:
            counts = ndimage.sum(mask, labeled, index=np.arange(1, num + 1))
            largest = int(np.argmax(counts)) + 1
            mask = (labeled == largest)

    return mask.astype(np.uint8)


# =========================================================
# 🧱 MARCHING CUBES + SUAVIZADO
# =========================================================
def mask_to_mesh(mask, spacing, step_size, smooth_iterations):
    if np.sum(mask) == 0:
        raise ValueError("Máscara vacía.")

    verts, faces, normals, _ = measure.marching_cubes(
        volume=mask.astype(np.float32),
        level=0.5,
        spacing=spacing,
        step_size=step_size
    )

    mesh = trimesh.Trimesh(vertices=verts, faces=faces, process=True)

    if smooth_iterations > 0:
        trimesh.smoothing.filter_laplacian(mesh, iterations=smooth_iterations)

    return mesh


# =========================================================
# 🎯 GENERAR MALLAS
# =========================================================
def generate_meshes_for_patient(nrrd_path, out_dir, labels=None, output_ext=".stl"):
    os.makedirs(out_dir, exist_ok=True)

    vol, spacing = load_nrrd_volume(nrrd_path)
    print(f"\nVolumen: {vol.shape} | Spacing: {spacing}")
    print(f"Clases presentes: {np.unique(vol).tolist()}\n")

    if labels is None:
        labels = list(LABEL_TO_NAME.keys())

    results = []

    for label in labels:
        name   = LABEL_TO_NAME.get(label, f"clase_{label}")
        config = LABEL_CONFIG.get(label, dict(
            min_size=200, keep_largest_cc=False, fill_holes=True,
            closing_radius=1, smooth_iterations=8, step_size=1
        ))

        print(f"[{label:02d}] {name} ...", end=" ", flush=True)

        mask  = (vol == label).astype(np.uint8)
        n_vox = int(mask.sum())

        if n_vox == 0:
            print("⚠️  Sin voxels, saltando.")
            continue

        try:
            mask_pp = postprocess_mask(
                mask            = mask,
                min_size        = config["min_size"],
                keep_largest_cc = config["keep_largest_cc"],
                fill_holes      = config["fill_holes"],
                closing_radius  = config["closing_radius"],
            )

            if mask_pp.sum() == 0:
                print("⚠️  Máscara vacía tras postprocesado, saltando.")
                continue

            mesh = mask_to_mesh(
                mask              = mask_pp,
                spacing           = spacing,
                step_size         = config["step_size"],
                smooth_iterations = config["smooth_iterations"],
            )

            out_path = os.path.join(out_dir, f"{label:02d}_{name}{output_ext}")
            mesh.export(out_path)

            print(f"✅  verts={len(mesh.vertices):,} | faces={len(mesh.faces):,} → {os.path.basename(out_path)}")
            results.append((label, name, out_path, len(mesh.vertices), len(mesh.faces)))

        except Exception as e:
            print(f"❌  Error: {e}")

    print(f"\n🏁 Mallas generadas: {len(results)}/{len(labels)}")
    return results


# =========================================================
# ▶️ EJECUCIÓN
# =========================================================
if __name__ == "__main__":
    generate_meshes_for_patient(
        nrrd_path  = NRRD_PATH,
        out_dir    = OUT_DIR,
        labels     = LABELS_TO_PROCESS,
        output_ext = ".stl"
    )